# Validar o CLIP quantizado (ONNX, 89 MB) antes de usar no site

O review sugere trocar o `vision_model.onnx` (352 MB) pelo `vision_model_quantized.onnx` (89 MB), **depois de validar com `ferramentas/validar_onnx.py`**.
Este notebook roda essa validação no Colab (o PyTorch não cabe na Vercel, mas aqui ele existe).

- Meta (`docs/decisoes.md`, seção 4): **r > 0,98** nas previsões de humor e energia, contra o CLIP em PyTorch.
- Se o quantizado passar: em `site/api/index.py`, trocar `MODELO_CLIP = "fp32"` por `MODELO_CLIP = "quantizado"`.
- Se não passar: fica o fp32, que já está no ar.

Como rodar: *Ambiente de execução → Executar tudo*.

In [ ]:
# 1. código do repositório e dependências
import subprocess
from pathlib import Path
REPO = Path("/content/Projeto-musical-Vibez")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/SofiaLimar/Projeto-musical-Vibez", str(REPO)], check=True)
!pip -q install onnxruntime huggingface_hub

In [ ]:
# 2. os dois modelos, na mesma versão fixada que o site usa, conferindo tamanho e sha256
import hashlib, re
from huggingface_hub import hf_hub_download

fonte = (REPO / "site" / "api" / "index.py").read_text()
REVISAO = re.search(r'"([0-9a-f]{40})/onnx/"', fonte).group(1)
ESPERADO = {nome: (int(tam.replace("_", "")), sha) for nome, tam, sha in
            re.findall(r'"(vision_model\w*\.onnx)", ([\d_]+),\s*"(\w{64})"', fonte)}
print("revisão", REVISAO, "|", ESPERADO.keys())

PASTA_ONNX = Path("/content/onnx")
for nome, (tamanho, sha) in ESPERADO.items():
    arq = Path(hf_hub_download("Xenova/clip-vit-base-patch32", f"onnx/{nome}", revision=REVISAO, local_dir=PASTA_ONNX))
    h = hashlib.sha256(arq.read_bytes()).hexdigest()
    print(f"{nome}: {arq.stat().st_size:,} bytes (esperado {tamanho:,}) | sha256 confere: {h == sha}")
    assert arq.stat().st_size == tamanho and h == sha, "arquivo diferente do configurado no site"


In [ ]:
# 3. fotos para a validação: 200 imagens do OASIS sorteadas (a parte 1 já baixou o OASIS no Drive)
import random, shutil
from google.colab import drive
drive.mount("/content/drive")
IMAGENS_OASIS = Path("/content/drive/MyDrive/mapa-sonoro/oasis/Images")   # mesma pasta da parte 1
FOTOS = Path("/content/fotos_validacao")
FOTOS.mkdir(exist_ok=True)
todas = sorted(IMAGENS_OASIS.glob("*.jpg"))
assert todas, f"Não achei imagens em {IMAGENS_OASIS}. Rode a célula do OASIS da parte 1 ou ajuste o caminho."
for p in random.Random(42).sample(todas, min(200, len(todas))):
    shutil.copy(p, FOTOS / p.name)
print(len(list(FOTOS.iterdir())), "fotos")

In [ ]:
# 4. a validação (cada modelo contra o CLIP em PyTorch)
%cd /content/Projeto-musical-Vibez
!python ferramentas/validar_onnx.py /content/onnx /content/fotos_validacao

## Como ler
Para cada arquivo aparece o cosseno mínimo entre os embeddings e o **r de humor e de energia** contra o PyTorch.
O quantizado passa se os dois r forem **maiores que 0,98**. Anote o resultado em `docs/decisoes.md` (seção 4) junto com a troca.